# Student Analytics Pipeline

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 45 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## Problem and outcome

A program lead needs a trustworthy cohort health summary from a small student database. The system fans out approved metrics, executes only allow-listed SQL, joins correlated results, and publishes one deterministic report with a complete Spore trail.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    get_events, require_env, setup_case_study, show_callout,
    show_flow, show_json, show_roles, show_spore, show_timeline, stage,
)

setup_case_study('Student Analytics Pipeline')


## Course prerequisites

This capstone assumes: `course-02-research-pipeline`, `course-04-parallel-agents`, `course-05-tool-use`. The implementation focuses on system design instead of explaining routine Python syntax.


## Architecture and responsibilities

A planner converts the bounded question into metric requests. An executor owns the database tool and its SQL allow-list. An aggregator waits for every expected metric. A reporter accepts the terminal cohort summary.


In [ ]:
show_roles([
('Planner', 'Choose approved metrics', 'agent'),
('SQL tool', 'Execute allow-listed queries', 'tool'),
('Aggregator', 'Join by analysis ID', 'agent'),
('Reporter', 'Publish one summary', 'agent')
])


## Design decisions

- The planner sends metric names, never free-form SQL.
- The tool maps names to fixed parameter-free SELECT statements.
- Aggregation checks the expected metric set, not only a result count.
- Every Spore retains the analysis ID for concurrent-run isolation.


## Implementation

The next cells are grouped by subsystem. Each group exposes the Praval objects that matter to the architecture.


In [ ]:
import sqlite3
import tempfile
import threading

from praval import agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

reset_reef()
reset_tool_registry()
workspace = tempfile.TemporaryDirectory(prefix="praval-student-analytics-")
database = str(Path(workspace.name) / "students.sqlite3")
connection = sqlite3.connect(database, check_same_thread=False)
connection.execute(
    "CREATE TABLE outcomes (student TEXT, cohort TEXT, score REAL, completed INTEGER)"
)
connection.executemany(
    "INSERT INTO outcomes VALUES (?, ?, ?, ?)",
    [
        ("Ada", "alpha", 91.0, 1), ("Ben", "alpha", 83.0, 1),
        ("Cy", "beta", 64.0, 0), ("Dee", "beta", 72.0, 1),
    ],
)
connection.commit()
database_lock = threading.Lock()


@tool(
    "case_student_metric", description="Run one allow-listed student metric.",
    category="analytics", shared=True,
)
def student_metric(metric: str) -> dict:
    queries = {
        "average_score": "SELECT cohort, ROUND(AVG(score), 1) FROM outcomes GROUP BY cohort",
        "completion_rate": "SELECT cohort, ROUND(AVG(completed) * 100, 1) FROM outcomes GROUP BY cohort",
    }
    if metric not in queries:
        raise ValueError("metric is not allow-listed")
    with database_lock:
        rows = connection.execute(queries[metric]).fetchall()
    return {cohort: value for cohort, value in rows}


### Planning and execution

Specialists exchange bounded metric names and correlated results.


In [ ]:
expected_metrics = {"average_score", "completion_rate"}
result_buckets = {}
completed_analyses = set()
reports = []
message_trail = []
aggregation_lock = threading.Lock()


@agent("case-analytics-planner", responds_to=["analytics_question"], auto_broadcast=False)
def planner(spore):
    message_trail.append(spore)
    for metric in sorted(expected_metrics):
        broadcast({
            "type": "metric_request", "analysis_id": spore.knowledge["analysis_id"],
            "metric": metric,
        })
    stage("planner", "metrics requested", ", ".join(sorted(expected_metrics)), spore)


@agent(
    "case-analytics-executor", responds_to=["metric_request"],
    tools=["case_student_metric"], auto_broadcast=False,
)
def executor(spore):
    message_trail.append(spore)
    metric = spore.knowledge["metric"]
    result = student_metric(metric)
    broadcast({
        "type": "metric_result", "analysis_id": spore.knowledge["analysis_id"],
        "metric": metric, "result": result,
    })
    stage("executor", "metric complete", metric, spore)


### Aggregation and reporting

The lock makes the join exactly once.

## End-to-end run

One question requests both approved metrics and waits for the terminal report.


In [ ]:
@agent("case-analytics-aggregator", responds_to=["metric_result"], auto_broadcast=False)
def aggregator(spore):
    message_trail.append(spore)
    analysis_id = spore.knowledge["analysis_id"]
    with aggregation_lock:
        bucket = result_buckets.setdefault(analysis_id, {})
        bucket[spore.knowledge["metric"]] = spore.knowledge["result"]
        ready = set(bucket) == expected_metrics and analysis_id not in completed_analyses
        if ready:
            completed_analyses.add(analysis_id)
            summary = dict(bucket)
    if ready:
        broadcast({"type": "analytics_report", "analysis_id": analysis_id,
                   "summary": summary})


@agent("case-analytics-reporter", responds_to=["analytics_report"], auto_broadcast=False)
def reporter(spore):
    message_trail.append(spore)
    reports.append(spore.knowledge)
    stage("reporter", "report complete", spore.knowledge["analysis_id"], spore)


start_agents(
    planner, executor, aggregator, reporter,
    initial_data={"type": "analytics_question", "analysis_id": "cohort-001",
                  "question": "Compare cohort performance."},
    channel="case-student-analytics",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert len(reports) == 1
assert set(reports[0]["summary"]) == expected_metrics
assert reports[0]["summary"]["average_score"] == {"alpha": 87.0, "beta": 68.0}


## Inspect the system

Inspect the evidence left by the completed run.


In [ ]:
show_json(reports[0], "Deterministic cohort report")
show_json(
    [{"type": item.knowledge["type"], "id": item.id,
      "analysis_id": item.knowledge["analysis_id"]}
     for item in message_trail],
    "Correlated Spore trail",
)
show_json({"tool": student_metric.get_tool_info(), "allowed": sorted(expected_metrics)},
          "SQL tool boundary")
show_timeline()


## Failure modes and tradeoffs

- The fixed schema is appropriate for a teaching fixture; production migrations and read-only credentials are still required.
- A metric allow-list prevents arbitrary SQL but must be reviewed when definitions change.
- The aggregation waits indefinitely if a specialist never emits a terminal result; production policy needs timeout and partial-failure handling.
- Thread locking protects this process only; distributed aggregation needs a shared transactional store.


## Extensions

- Add an allow-listed attendance metric and update the expected set.
- Send two analysis IDs concurrently and verify their result buckets never mix.
- Emit an explicit failed metric result and design a degraded report policy.


## Cleanup

Release project resources explicitly.


In [ ]:
connection.close()
workspace.cleanup()
for function in (planner, executor, aggregator, reporter):
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
